In [78]:
import os
import csv
import ast
import re
import time
import nltk
import collections
import numpy as np
from sklearn.model_selection import train_test_split, GridSearchCV, KFold
from sklearn.metrics import accuracy_score, hamming_loss, precision_recall_fscore_support
from sklearn.preprocessing import MultiLabelBinarizer
from sklearn.linear_model import LogisticRegression
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.multiclass import OneVsRestClassifier
from sklearn.svm import LinearSVC

In [79]:
# Variables globales:
SEED = 2627
DEV_PART = 0.15
CORPUS_DIR = "data"
TRAIN_FILE = os.path.join(CORPUS_DIR, "dataset_train.csv")
TEST_FILE = os.path.join(CORPUS_DIR, "dataset_test.csv")
TOKEN_PATTERN = r"\S+" # Captura cualquier grupo de caracteres que NO contenga espacios

In [80]:
# Función para cargar el corpus:
def load_corpus(filename, delimiter=','):
    papers, terms = [], []

    # Abrimos el archivo 'filename' y lo leemos línea por línea:
    with open(filename, mode='r', encoding='utf-8') as file:
        reader = csv.reader(file, delimiter=delimiter)

        # Saltamos la fila con los títulos de las columnas:
        next(reader)

        # Extraemos las columnas del archivo:
        for row in reader:
            if not row or len(row) < 3:
                continue

            papers.append((row[0], row[1]))
            terms.append(ast.literal_eval(row[-1]))

    return papers, terms


# Función de tokenización:
def tokenize(text):
    # Buscamos palabras que empiecen por letras y puedan contener guiones
    # (para evitar ruido como "5k" y no excluir estructuras como "state-of-the-art"):
    pattern = re.compile(r"(?u)\b[a-zA-Z][\w-]+\b")
    return " ".join(pattern.findall(text))


# Función para calcular las métricas de evaluación:
def compute_metrics(labels, preds):
    precision, recall, f1, _ = precision_recall_fscore_support(
        labels, preds, average="macro", zero_division=0
    )
    acc = accuracy_score(labels, preds)
    hl = hamming_loss(labels, preds)

    return {
        "accuracy": acc,
        "f1": f1,
        "precision": precision,
        "recall": recall,
        "hamming_loss": hl,
    }

def show_stats(texts, labels, name):
    print(f"{name}\n" + "-" * len(name))
    ntok = [len(t.split()) for t in texts]
    print(f"Muestras: {len(texts)}   Tokens: {sum(ntok)}   Tokens por muestra: {np.mean(ntok):.2f}")
    ct = collections.Counter(labels)
    print("  ".join(f"{k}: {ct[k]} ({100 * ct[k] / len(labels):.1f} %)" for k in sorted(ct)))
    print()

try:
    stop_words = nltk.corpus.stopwords.words('english')
except LookupError:
    nltk.download('stopwords')
    stop_words = nltk.corpus.stopwords.words('english')

In [81]:
# Cargamos el corpus, y guardamos las entradas y los términos:
raw_all, y_all = load_corpus(TRAIN_FILE)
print("Papers de entrenamiento:", len(raw_all))

# Convertimos los términos en listas binarias:
mlb = MultiLabelBinarizer()
y_bin = mlb.fit_transform(y_all)
print(mlb.classes_)

# Tokenizamos los títulos y los resúmenes:
raw_tok = [tokenize(title) + tokenize(summary) for title, summary in raw_all]

# Creamos una partición aleatoria de entrenamiento/desarrollo:
train_tok, dev_tok, y_train, y_dev = train_test_split(raw_tok, y_bin, test_size=DEV_PART, shuffle=True, random_state=SEED)

Papers de entrenamiento: 43555
['cs.AI' 'cs.CL' 'cs.CR' 'cs.CV' 'cs.CY' 'cs.DC' 'cs.GR' 'cs.HC' 'cs.IR'
 'cs.IT' 'cs.LG' 'cs.MA' 'cs.MM' 'cs.NE' 'cs.RO' 'cs.SI' 'cs.SY' 'eess.IV'
 'eess.SP' 'eess.SY' 'math.IT' 'math.OC' 'math.ST' 'physics.chem-ph'
 'stat.AP' 'stat.ME' 'stat.ML' 'stat.TH']


In [61]:
labels_train = [cat for paper_cats in mlb.inverse_transform(y_train) for cat in paper_cats]
labels_dev = [cat for paper_cats in mlb.inverse_transform(y_dev) for cat in paper_cats]

show_stats(train_tok, labels_train, "Entrenamiento")
show_stats(dev_tok, labels_dev, "Validación")

Entrenamiento
-------------
Muestras: 37021   Tokens: 6484882   Tokens por muestra: 175.17
cs.AI: 5137 (8.0 %)  cs.CL: 901 (1.4 %)  cs.CR: 283 (0.4 %)  cs.CV: 23005 (35.7 %)  cs.CY: 82 (0.1 %)  cs.DC: 86 (0.1 %)  cs.GR: 288 (0.4 %)  cs.HC: 38 (0.1 %)  cs.IR: 153 (0.2 %)  cs.IT: 44 (0.1 %)  cs.LG: 19222 (29.9 %)  cs.MA: 186 (0.3 %)  cs.MM: 237 (0.4 %)  cs.NE: 525 (0.8 %)  cs.RO: 1142 (1.8 %)  cs.SI: 358 (0.6 %)  cs.SY: 74 (0.1 %)  eess.IV: 1527 (2.4 %)  eess.SP: 253 (0.4 %)  eess.SY: 74 (0.1 %)  math.IT: 44 (0.1 %)  math.OC: 246 (0.4 %)  math.ST: 37 (0.1 %)  physics.chem-ph: 31 (0.0 %)  stat.AP: 52 (0.1 %)  stat.ME: 44 (0.1 %)  stat.ML: 10281 (16.0 %)  stat.TH: 37 (0.1 %)

Validación
----------
Muestras: 6534   Tokens: 1145100   Tokens por muestra: 175.25
cs.AI: 899 (7.9 %)  cs.CL: 151 (1.3 %)  cs.CR: 45 (0.4 %)  cs.CV: 4073 (35.8 %)  cs.CY: 18 (0.2 %)  cs.DC: 16 (0.1 %)  cs.GR: 50 (0.4 %)  cs.HC: 10 (0.1 %)  cs.IR: 16 (0.1 %)  cs.IT: 5 (0.0 %)  cs.LG: 3379 (29.7 %)  cs.MA: 29 (0.3 %)  

# Dummy Classifier #

In [38]:
print(compute_metrics(y_dev, np.zeros_like(y_dev)))
print(compute_metrics(y_dev, np.ones_like(y_dev)))

{'accuracy': 0.0, 'f1': 0.0, 'precision': 0.0, 'recall': 0.0, 'hamming_loss': 0.06210918710918711}
{'accuracy': 0.0, 'f1': 0.08888139346828376, 'precision': 0.062109187109187114, 'recall': 1.0, 'hamming_loss': 0.9378908128908129}


# Logistic Regression #

In [59]:
vec = TfidfVectorizer(token_pattern=TOKEN_PATTERN, sublinear_tf=True)
A = vec.fit_transform(train_tok)
B = vec.transform(dev_tok)

In [ ]:
multilabel = {}
for c in [0.1, 0.25, 0.5, 0.75, 1, 3, 5, 6, 7, 8, 9, 10]:
    multilabel[c] = OneVsRestClassifier(LogisticRegression(solver="liblinear", C=c, class_weight="balanced", random_state=SEED)).fit(A, y_train)
    y_pred = multilabel[c].predict(B)
    print(f"=== C={c}")
    print(compute_metrics(y_dev, y_pred))

=== C=0.1, 0.1
{'accuracy': 0.25068870523415976, 'f1': 0.3828775210093977, 'precision': 0.31286728597454694, 'recall': 0.6942435956382225, 'hamming_loss': 0.046564126109580654}
=== C=0.25, 0.25
{'accuracy': 0.3111417202326293, 'f1': 0.39915850688501014, 'precision': 0.33524251235804814, 'recall': 0.6494154681800712, 'hamming_loss': 0.04026192662556299}
=== C=0.5, 0.5
{'accuracy': 0.37327823691460055, 'f1': 0.43541858424590946, 'precision': 0.38825915061601834, 'recall': 0.6411783595984153, 'hamming_loss': 0.03542459224277406}
=== C=0.75, 0.75
{'accuracy': 0.4097030915212733, 'f1': 0.4547737544094469, 'precision': 0.41272095939201936, 'recall': 0.6301159019055236, 'hamming_loss': 0.0327517600244873}
=== C=1, 1
{'accuracy': 0.43357820630547905, 'f1': 0.4639510064438256, 'precision': 0.4297905105335521, 'recall': 0.6102667049626519, 'hamming_loss': 0.030969871878962787}
=== C=3, 3
{'accuracy': 0.5238751147842057, 'f1': 0.5138451094128209, 'precision': 0.5954641341570015, 'recall': 0.57326

In [ ]:
for c100 in range(100, 300, 25):
    c = c100 / 100
    multilabel[c] = OneVsRestClassifier(LogisticRegression(solver="liblinear", C=c, class_weight="balanced", random_state=SEED)).fit(A, y_train)
    y_pred = multilabel[c].predict(B)
    print(f"=== C={c}")
    print(compute_metrics(y_dev, y_pred))

=== C=1.0, 1.0
{'accuracy': 0.43357820630547905, 'f1': 0.4639510064438256, 'precision': 0.4297905105335521, 'recall': 0.6102667049626519, 'hamming_loss': 0.030969871878962787}
=== C=1.25, 1.25
{'accuracy': 0.45454545454545453, 'f1': 0.4741469755451201, 'precision': 0.4488947523502235, 'recall': 0.6001223513153785, 'hamming_loss': 0.02954873409418864}
=== C=1.5, 1.5
{'accuracy': 0.465564738292011, 'f1': 0.47866905726333486, 'precision': 0.4564828616775423, 'recall': 0.5959096612613257, 'hamming_loss': 0.028657790021426384}
=== C=1.75, 1.75
{'accuracy': 0.47796143250688705, 'f1': 0.489565304352822, 'precision': 0.47838506882445003, 'recall': 0.5931420461969061, 'hamming_loss': 0.02784336875245966}
=== C=2.0, 2.0
{'accuracy': 0.48806244260789716, 'f1': 0.49213598038279266, 'precision': 0.48277473410530447, 'recall': 0.5889316842891587, 'hamming_loss': 0.02717106126197035}
=== C=2.25, 2.25
{'accuracy': 0.4989286807468626, 'f1': 0.497357754397144, 'precision': 0.49667765169345357, 'recall':

In [ ]:
for c100 in range(250, 275, 5):
    c = c100 / 100
    multilabel[c] = OneVsRestClassifier(LogisticRegression(solver="liblinear", C=c, class_weight="balanced", random_state=SEED)).fit(A, y_train)
    y_pred = multilabel[c].predict(B)
    print(f"=== C={c}")
    print(compute_metrics(y_dev, y_pred))

=== C=2.5, 2.5
{'accuracy': 0.5071931435567799, 'f1': 0.500750922198366, 'precision': 0.5140615258117424, 'recall': 0.5790022862295928, 'hamming_loss': 0.026077878350605623}
=== C=2.55, 2.55
{'accuracy': 0.509641873278237, 'f1': 0.5037874182855876, 'precision': 0.5266271249278097, 'recall': 0.5786663224542475, 'hamming_loss': 0.025946696401241857}
=== C=2.6, 2.6
{'accuracy': 0.5113253749617386, 'f1': 0.5061169881979766, 'precision': 0.5514700010458129, 'recall': 0.5787049985127677, 'hamming_loss': 0.025842844024662207}
=== C=2.65, 2.65
{'accuracy': 0.512243648607285, 'f1': 0.5104979213073516, 'precision': 0.588563349393221, 'recall': 0.5784142118365505, 'hamming_loss': 0.02575538939175303}
=== C=2.7, 2.7
{'accuracy': 0.5139271502907866, 'f1': 0.5110570925513228, 'precision': 0.589402790042684, 'recall': 0.5780076089870085, 'hamming_loss': 0.025646071100616556}


In [25]:
vec_min_df = TfidfVectorizer(token_pattern=TOKEN_PATTERN, sublinear_tf=True, min_df=2)
A_min_df = vec_min_df.fit_transform(train_tok)
B_min_df = vec_min_df.transform(dev_tok)
model = OneVsRestClassifier(LogisticRegression(solver="liblinear", C=2.7, class_weight="balanced", random_state=SEED)).fit(A_min_df, y_train)
y_pred = model.predict(B_min_df)
print(f"=== C=2.7, min_df=2")
print(compute_metrics(y_dev, y_pred))

=== C=2.7, min_df=2
{'accuracy': 0.49173553719008267, 'f1': 0.4977696632073387, 'precision': 0.5092033392407032, 'recall': 0.5781790265052272, 'hamming_loss': 0.02692509510691329}


In [60]:
vec_min_df = TfidfVectorizer(token_pattern=TOKEN_PATTERN, sublinear_tf=True, min_df=2)
A_min_df = vec_min_df.fit_transform(train_tok)
B_min_df = vec_min_df.transform(dev_tok)
model = OneVsRestClassifier(LogisticRegression(solver="liblinear", C=0.5, class_weight="balanced", random_state=SEED)).fit(A_min_df, y_train)
y_pred = model.predict(B_min_df)
print(f"=== C=0.5, min_df=2")
print(compute_metrics(y_dev, y_pred))

=== C=0.5, min_df=2
{'accuracy': 0.3598102234465871, 'f1': 0.42979292273813563, 'precision': 0.3663382533004419, 'recall': 0.6438888116203242, 'hamming_loss': 0.03640299094844549}


### Con StopWords ###

In [12]:
vec_stp = TfidfVectorizer(token_pattern=TOKEN_PATTERN, sublinear_tf=True, stop_words=stop_words)
A_stp = vec_stp.fit_transform(train_tok)
B_stp = vec_stp.transform(dev_tok)

In [ ]:
multilabel_stp = {}
for c in [3.25, 3.5, 3.75, 4, 4.5, 4.75]:
    multilabel_stp[c] = OneVsRestClassifier(LogisticRegression(solver="liblinear", C=c, class_weight="balanced", random_state=SEED)).fit(A_stp, y_train)
    y_pred = multilabel_stp[c].predict(B_stp)
    print(f"=== C={c}, stopwords")
    print(compute_metrics(y_dev, y_pred))

=== C=3.25, stopwords
{'accuracy': 0.5330578512396694, 'f1': 0.5057294534191676, 'precision': 0.5360329966038695, 'recall': 0.5604182301042485, 'hamming_loss': 0.02457475184747912}
=== C=3.5, stopwords
{'accuracy': 0.5387205387205387, 'f1': 0.5108357519300324, 'precision': 0.5651158863095064, 'recall': 0.5602812317613314, 'hamming_loss': 0.02421400148672876}
=== C=3.75, stopwords
{'accuracy': 0.5436179981634527, 'f1': 0.5117229643060929, 'precision': 0.5675640975254093, 'recall': 0.5583473212986494, 'hamming_loss': 0.023951637588001225}
=== C=4, stopwords
{'accuracy': 0.546985001530456, 'f1': 0.5139670332072098, 'precision': 0.5708089102515047, 'recall': 0.5578132102168694, 'hamming_loss': 0.0237384669202851}
=== C=4.5, stopwords
{'accuracy': 0.5546372819100092, 'f1': 0.5162675380473041, 'precision': 0.5768324626884012, 'recall': 0.5538031201363017, 'hamming_loss': 0.023333989243080153}
=== C=4.75, stopwords
{'accuracy': 0.5576981940618304, 'f1': 0.5175639054400755, 'precision': 0.5830

In [26]:
vec_min_df_stp = TfidfVectorizer(token_pattern=TOKEN_PATTERN, sublinear_tf=True, min_df=2, stop_words=stop_words)
A_min_df_stp = vec_min_df.fit_transform(train_tok)
B_min_df_stp = vec_min_df.transform(dev_tok)
model = OneVsRestClassifier(LogisticRegression(solver="liblinear", C=2.7, class_weight="balanced", random_state=SEED)).fit(A_min_df_stp, y_train)
y_pred = model.predict(B_min_df_stp)
print(f"=== C=2.7, min_df=2")
print(compute_metrics(y_dev, y_pred))

=== C=2.7, min_df=2
{'accuracy': 0.49173553719008267, 'f1': 0.4977696632073387, 'precision': 0.5092033392407032, 'recall': 0.5781790265052272, 'hamming_loss': 0.02692509510691329}


### Con ngramas ###

In [8]:
ngrams = {}
for n in [(1, 2), (1, 3)]:
    ngram_vec = TfidfVectorizer(token_pattern=TOKEN_PATTERN, ngram_range=n, sublinear_tf=True)
    A_ngram = ngram_vec.fit_transform(train_tok)
    B_ngram = ngram_vec.transform(dev_tok)
    ngrams[n] = OneVsRestClassifier(LogisticRegression(solver="liblinear", C=2.7, class_weight="balanced", random_state=SEED)).fit(A_ngram, y_train)
    y_pred = ngrams[n].predict(B_ngram)
    print(f"=== ngrams {n}")
    print(compute_metrics(y_dev, y_pred))
    print(f"Rasgos: {A_ngram.shape[1]} n-gramas frente a {A.shape[1]} con palabras")

=== ngrams (1, 2)
{'accuracy': 0.62825221916131, 'f1': 0.5480403762830971, 'precision': 0.7333476204510374, 'recall': 0.5147052461866057, 'hamming_loss': 0.019125235034325942}
Rasgos: 1215718 n-gramas frente a 91157 con palabras
=== ngrams (1, 3)
{'accuracy': 0.6533516988062442, 'f1': 0.5388694215108524, 'precision': 0.7612697702773551, 'recall': 0.4892871022264565, 'hamming_loss': 0.017928199746381563}
Rasgos: 4231721 n-gramas frente a 91157 con palabras


### Con ngramas y StopWords ###

In [12]:
ngram_vec = TfidfVectorizer(token_pattern=TOKEN_PATTERN, ngram_range=(1, 2), sublinear_tf=True, stop_words=stop_words)
A_ngram = ngram_vec.fit_transform(train_tok)
B_ngram = ngram_vec.transform(dev_tok)
ngrams = OneVsRestClassifier(LogisticRegression(solver="liblinear", C=2.7, class_weight="balanced", random_state=SEED)).fit(A_ngram, y_train)
y_pred = ngrams.predict(B_ngram)
print(f"=== ngrams (1, 2) stopwords")
print(compute_metrics(y_dev, y_pred))
print(f"Rasgos: {A_ngram.shape[1]} n-gramas frente a {A.shape[1]} con palabras")

=== ngrams (1, 2) stopwords
{'accuracy': 0.6296296296296297, 'f1': 0.5533394139982376, 'precision': 0.7276205102028442, 'recall': 0.5315313147770129, 'hamming_loss': 0.01898312125584853}
Rasgos: 1554041 n-gramas frente a 91157 con palabras


In [14]:
ngram_vec = TfidfVectorizer(token_pattern=TOKEN_PATTERN, ngram_range=(1, 2), sublinear_tf=True, stop_words=stop_words, min_df=2)
A_ngram = ngram_vec.fit_transform(train_tok)
B_ngram = ngram_vec.transform(dev_tok)
ngrams = OneVsRestClassifier(LogisticRegression(solver="liblinear", C=2.7, class_weight="balanced", random_state=SEED)).fit(A_ngram, y_train)
y_pred = ngrams.predict(B_ngram)
print(f"=== ngrams (1, 2) stopwords")
print(compute_metrics(y_dev, y_pred))
print(f"Rasgos: {A_ngram.shape[1]} n-gramas frente a {A.shape[1]} con palabras")

=== ngrams (1, 2) stopwords
{'accuracy': 0.6118763391490665, 'f1': 0.5405876640219832, 'precision': 0.6945087667053353, 'recall': 0.5304406503293799, 'hamming_loss': 0.019994315448860902}
Rasgos: 638782 n-gramas frente a 91157 con palabras


In [15]:
ngram_vec = TfidfVectorizer(token_pattern=TOKEN_PATTERN, ngram_range=(1, 2), sublinear_tf=True, min_df=2)
A_ngram = ngram_vec.fit_transform(train_tok)
B_ngram = ngram_vec.transform(dev_tok)
ngrams = OneVsRestClassifier(LogisticRegression(solver="liblinear", C=2.7, class_weight="balanced", random_state=SEED)).fit(A_ngram, y_train)
y_pred = ngrams.predict(B_ngram)
print(f"=== ngrams (1, 2) stopwords")
print(compute_metrics(y_dev, y_pred))
print(f"Rasgos: {A_ngram.shape[1]} n-gramas frente a {A.shape[1]} con palabras")

=== ngrams (1, 2) stopwords
{'accuracy': 0.6137128864401592, 'f1': 0.5472162383444925, 'precision': 0.7166845616498837, 'recall': 0.5213292653917103, 'hamming_loss': 0.019803008439372077}
Rasgos: 545903 n-gramas frente a 91157 con palabras


# Linear SVC #

In [ ]:
vectorizer = TfidfVectorizer(
    token_pattern=TOKEN_PATTERN,
    ngram_range=(1, 2),
    sublinear_tf=True,
    min_df=3,
    max_features=25000,
)
X_train = vectorizer.fit_transform(train_tok)
X_dev = vectorizer.transform(dev_tok)

for c in [0.05, 0.1, 0.2, 0.3, 0.5, 0.7, 1.0]:
    svm = OneVsRestClassifier(
        LinearSVC(
            C=c,
            dual=False,
            class_weight="balanced",
            max_iter=8000,
            tol=1e-3,
            random_state=SEED,
        ),
        n_jobs=-1,
    )
    svm.fit(X_train, y_train)

    y_pred = svm.predict(X_dev)
    metricas = compute_metrics(y_dev, y_pred)
    print(f"** C = {c}: **\n{metricas}")

** C = 0.05: **
{'accuracy': 0.40404040404040403, 'f1': 0.47312112402855305, 'precision': 0.554330646889048, 'recall': 0.5743006302336433, 'hamming_loss': 0.032735362280816825}
** C = 0.1: **
{'accuracy': 0.46250382614018976, 'f1': 0.5058987670087542, 'precision': 0.618609979135336, 'recall': 0.5510877521815656, 'hamming_loss': 0.02836809654991473}
** C = 0.2: **
{'accuracy': 0.5212733394551576, 'f1': 0.5349563173777655, 'precision': 0.6761771886066005, 'recall': 0.5334211456373648, 'hamming_loss': 0.02491910446455901}
** C = 0.3: **
{'accuracy': 0.5501989592898684, 'f1': 0.5445386083168936, 'precision': 0.7125392960204274, 'recall': 0.5163932885524561, 'hamming_loss': 0.02324106869561415}
** C = 0.5: **
{'accuracy': 0.581267217630854, 'f1': 0.5608064410877418, 'precision': 0.7710954956748013, 'recall': 0.5075629433886081, 'hamming_loss': 0.021727010363374}
** C = 0.7: **
{'accuracy': 0.6020814202632384, 'f1': 0.5618556733048241, 'precision': 0.7916977039900904, 'recall': 0.49732322031

In [14]:
for c in [0.6, 0.8, 0.9]:
    svm = OneVsRestClassifier(
        LinearSVC(
            C=c,
            dual=False,
            class_weight="balanced",
            max_iter=8000,
            tol=1e-3,
            random_state=SEED,
        ),
        n_jobs=-1,
    )
    svm.fit(X_train, y_train)

    y_pred = svm.predict(X_dev)
    metricas = compute_metrics(y_dev, y_pred)
    print(f"** C = {c}: **\n{metricas}")

** C = 0.6: **
{'accuracy': 0.5930517294153658, 'f1': 0.5652170919738506, 'precision': 0.7878031171363766, 'recall': 0.5044286884461882, 'hamming_loss': 0.02115855524946434}
** C = 0.8: **
{'accuracy': 0.608356290174472, 'f1': 0.563008414255246, 'precision': 0.7971403475019396, 'recall': 0.49498532381869326, 'hamming_loss': 0.020447986357077265}
** C = 0.9: **
{'accuracy': 0.6138659320477502, 'f1': 0.5607706079860381, 'precision': 0.8015712512535824, 'recall': 0.48809945307152686, 'hamming_loss': 0.020191088372906556}


In [15]:
for c in [0.6, 0.63, 0.66, 0.68]:
    svm = OneVsRestClassifier(
        LinearSVC(
            C=c,
            dual=False,
            class_weight="balanced",
            max_iter=8000,
            tol=1e-3,
            random_state=SEED,
        ),
        n_jobs=-1,
    )
    svm.fit(X_train, y_train)

    y_pred = svm.predict(X_dev)
    metricas = compute_metrics(y_dev, y_pred)
    print(f"** C = {c}: **\n{metricas}")

** C = 0.6: **
{'accuracy': 0.5930517294153658, 'f1': 0.5652170919738506, 'precision': 0.7878031171363766, 'recall': 0.5044286884461882, 'hamming_loss': 0.02115855524946434}
** C = 0.63: **
{'accuracy': 0.5962656871747781, 'f1': 0.5633874677167723, 'precision': 0.7889198981989681, 'recall': 0.5009058457697028, 'hamming_loss': 0.021016441470986927}
** C = 0.66: **
{'accuracy': 0.5982552800734618, 'f1': 0.5638386574042457, 'precision': 0.7900612542936413, 'recall': 0.5006799209452472, 'hamming_loss': 0.020885259521623158}
** C = 0.68: **
{'accuracy': 0.60085705540251, 'f1': 0.5613047035476034, 'precision': 0.7896924713700919, 'recall': 0.49759322401456874, 'hamming_loss': 0.020803270803270803}


# Pruebas con test #

In [69]:
raw_test, y_raw_test = load_corpus("data/etiquetas_test.csv", delimiter=';')

y_test = mlb.transform(y_raw_test)

test_tok = [tokenize(title) + tokenize(summary) for title, summary in raw_test]

labels_test = [cat for paper_cats in mlb.inverse_transform(y_test) for cat in paper_cats if cat in mlb.classes_]

show_stats(test_tok, labels_test, "Test")

TypeError: load_corpus() got an unexpected keyword argument 'delimiter'

In [23]:
vectorizer = TfidfVectorizer(
    token_pattern=TOKEN_PATTERN,
    ngram_range=(1, 2),
    sublinear_tf=True,
    min_df=3,
    max_features=25000,
)
X_train = vectorizer.fit_transform(train_tok)
X_test = vectorizer.transform(test_tok)

svm = OneVsRestClassifier(
    LinearSVC(
        C=0.6,
        dual=False,
        class_weight="balanced",
        max_iter=8000,
        tol=1e-3,
        random_state=SEED,
    ),
    n_jobs=-1,
)
svm.fit(X_train, y_train)
y_pred = svm.predict(X_test)
print("=== SVM bigramas, min_df=3, C=0.6")
print(compute_metrics(y_test, y_pred))

=== SVM bigramas, min_df=3, C=0.6
{'accuracy': 0.29723643807574207, 'f1': 0.1632922275787232, 'precision': 0.20673301816280024, 'recall': 0.186100467510424, 'hamming_loss': 0.03904810644831116}


In [24]:
ngram_vec = TfidfVectorizer(token_pattern=TOKEN_PATTERN, ngram_range=(1, 2), sublinear_tf=True, stop_words=stop_words)
A_ngram = ngram_vec.fit_transform(train_tok)
B_ngram = ngram_vec.transform(test_tok)
ngrams = OneVsRestClassifier(LogisticRegression(solver="liblinear", C=2.7, class_weight="balanced", random_state=SEED)).fit(A_ngram, y_train)
y_pred = ngrams.predict(B_ngram)
print(f"=== ngrams (1, 2) stopwords")
print(compute_metrics(y_test, y_pred))
print(f"Rasgos: {A_ngram.shape[1]} n-gramas frente a {A.shape[1]} con palabras")

=== ngrams (1, 2) stopwords
{'accuracy': 0.3142272262026612, 'f1': 0.16650769163140317, 'precision': 0.18817843734675913, 'recall': 0.21222073579915882, 'hamming_loss': 0.03779792367305162}
Rasgos: 1554041 n-gramas frente a 91157 con palabras


In [30]:
vec_stp = TfidfVectorizer(token_pattern=TOKEN_PATTERN, sublinear_tf=True, stop_words=stop_words)
A_stp = vec_stp.fit_transform(train_tok)
B_stp = vec_stp.transform(test_tok)

model = OneVsRestClassifier(LogisticRegression(solver="liblinear", C=4.6, class_weight="balanced", random_state=SEED)).fit(A_stp, y_train)
y_pred = model.predict(B_stp)
print(f"=== C=4.6, stopwords")
print(compute_metrics(y_test, y_pred))

=== C=4.6, stopwords
{'accuracy': 0.2786079836233367, 'f1': 0.18090583038039257, 'precision': 0.16874818201560315, 'recall': 0.24344541026490502, 'hamming_loss': 0.040576107618072815}


In [34]:
vectorizer = TfidfVectorizer(
    token_pattern=TOKEN_PATTERN,
    ngram_range=(1, 2),
    sublinear_tf=True,
    min_df=3,
    max_features=25000,
)
X_train = vectorizer.fit_transform(raw_tok)
X_test = vectorizer.transform(test_tok)

svm = OneVsRestClassifier(
    LinearSVC(
        C=0.6,
        dual=False,
        class_weight="balanced",
        max_iter=8000,
        tol=1e-3,
        random_state=SEED,
    ),
    n_jobs=-1,
)
svm.fit(X_train, y_bin)
y_pred = svm.predict(X_test)
print("=== SVM bigramas, min_df=3, C=0.6")
print(compute_metrics(y_test, y_pred))

=== SVM bigramas, min_df=3, C=0.6
{'accuracy': 0.29007164790174, 'f1': 0.16042597138540732, 'precision': 0.2010736506690292, 'recall': 0.18623518457573468, 'hamming_loss': 0.039450212019301066}


In [35]:
ngram_vec = TfidfVectorizer(token_pattern=TOKEN_PATTERN, ngram_range=(1, 2), sublinear_tf=True, stop_words=stop_words)
A_ngram = ngram_vec.fit_transform(raw_tok)
B_ngram = ngram_vec.transform(test_tok)
ngrams = OneVsRestClassifier(LogisticRegression(solver="liblinear", C=2.7, class_weight="balanced", random_state=SEED)).fit(A_ngram, y_bin)
y_pred = ngrams.predict(B_ngram)
print(f"=== ngrams (1, 2) stopwords")
print(compute_metrics(y_test, y_pred))
print(f"Rasgos: {A_ngram.shape[1]} n-gramas frente a {A.shape[1]} con palabras")

=== ngrams (1, 2) stopwords
{'accuracy': 0.3127942681678608, 'f1': 0.16590531042033627, 'precision': 0.18647922593532987, 'recall': 0.215182895176352, 'hamming_loss': 0.03797338792221085}
Rasgos: 1703642 n-gramas frente a 91157 con palabras


In [51]:
vec_stp = TfidfVectorizer(token_pattern=TOKEN_PATTERN, sublinear_tf=True, stop_words=stop_words)
A_stp = vec_stp.fit_transform(raw_tok)
B_stp = vec_stp.transform(test_tok)

for c in [4.6, 2, 1, 0.5, 0.1]:
    model = OneVsRestClassifier(LogisticRegression(solver="liblinear", C=c, class_weight="balanced", random_state=SEED)).fit(A_stp, y_bin)
    y_pred = model.predict(B_stp)
    print(f"=== C={c}, stopwords")
    print(compute_metrics(y_test, y_pred))

=== C=4.6, stopwords
{'accuracy': 0.27328556806550663, 'f1': 0.1790017279880791, 'precision': 0.16136586753593707, 'recall': 0.24708046065698694, 'hamming_loss': 0.041175610469366866}
=== C=2, stopwords
{'accuracy': 0.25465711361310134, 'f1': 0.18124414275137007, 'precision': 0.1550608383811276, 'recall': 0.27747846733041903, 'hamming_loss': 0.04334698055271238}
=== C=1, stopwords
{'accuracy': 0.22743091095189355, 'f1': 0.18303277965564108, 'precision': 0.15193132621895347, 'recall': 0.30990789809221236, 'hamming_loss': 0.04614709752887849}
=== C=0.5, stopwords
{'accuracy': 0.20184237461617197, 'f1': 0.18012414457024764, 'precision': 0.16180904388440018, 'recall': 0.3283018532817546, 'hamming_loss': 0.04915923380611201}
=== C=0.1, stopwords
{'accuracy': 0.15148413510747186, 'f1': 0.18005416355543594, 'precision': 0.14915703930875907, 'recall': 0.394397628340876, 'hamming_loss': 0.05753765170346542}


In [41]:
ngrams = OneVsRestClassifier(LogisticRegression(solver="liblinear", C=1, class_weight="balanced", random_state=SEED)).fit(A_ngram, y_bin)
y_pred = ngrams.predict(B_ngram)
print(f"=== ngrams (1, 2) stopwords")
print(compute_metrics(y_test, y_pred))
print(f"Rasgos: {A_ngram.shape[1]} n-gramas frente a {A.shape[1]} con palabras")

=== ngrams (1, 2) stopwords
{'accuracy': 0.2775844421699079, 'f1': 0.17947369802484772, 'precision': 0.19150224110950492, 'recall': 0.25654142257306606, 'hamming_loss': 0.04141687381196081}
Rasgos: 1703642 n-gramas frente a 91157 con palabras


In [52]:
vec = TfidfVectorizer(token_pattern=TOKEN_PATTERN, sublinear_tf=True)
A = vec.fit_transform(raw_tok)
B = vec.transform(test_tok)

for c in [4.6, 2, 1, 0.5, 0.1]:
    model = OneVsRestClassifier(LogisticRegression(solver="liblinear", C=c, class_weight="balanced", random_state=SEED)).fit(A, y_bin)
    y_pred = model.predict(B)
    print(f"=== C={c}")
    print(compute_metrics(y_test, y_pred))

=== C=4.6
{'accuracy': 0.2724667349027636, 'f1': 0.17748696203215344, 'precision': 0.1570940529181726, 'recall': 0.24773917490560196, 'hamming_loss': 0.041263342593946484}
=== C=2
{'accuracy': 0.24564994882292732, 'f1': 0.1777876100918214, 'precision': 0.1505742158452383, 'recall': 0.2803600934448222, 'hamming_loss': 0.04403421552858605}
=== C=1
{'accuracy': 0.2202661207778915, 'f1': 0.18143385934232784, 'precision': 0.1498775670118937, 'recall': 0.3121678509086302, 'hamming_loss': 0.04679046644246235}
=== C=0.5
{'accuracy': 0.19611054247697032, 'f1': 0.18740309920498505, 'precision': 0.16856578667440192, 'recall': 0.3436389191772416, 'hamming_loss': 0.05001462202076327}
=== C=0.1
{'accuracy': 0.14411463664278404, 'f1': 0.18053587935467935, 'precision': 0.14759607773638814, 'recall': 0.4039372236121996, 'hamming_loss': 0.05826144173124726}


In [53]:
vec_min_df = TfidfVectorizer(token_pattern=TOKEN_PATTERN, sublinear_tf=True, min_df=2)
A = vec_min_df.fit_transform(raw_tok)
B = vec_min_df.transform(test_tok)

for c in [4.6, 2.7, 2, 1, 0.5, 0.3, 0.1]:
    model = OneVsRestClassifier(LogisticRegression(solver="liblinear", C=c, class_weight="balanced", random_state=SEED)).fit(A, y_bin)
    y_pred = model.predict(B)
    print(f"=== C={c}, min_df=2")
    print(compute_metrics(y_test, y_pred))

=== C=4.6, min_df=2
{'accuracy': 0.2632548618219038, 'f1': 0.1787899550794296, 'precision': 0.1557802528011384, 'recall': 0.2554300490287427, 'hamming_loss': 0.04218452990203246}
=== C=2.7, min_df=2
{'accuracy': 0.2483111566018424, 'f1': 0.17994803195327513, 'precision': 0.15400322075500816, 'recall': 0.27458508300154, 'hamming_loss': 0.04362479894721451}
=== C=2, min_df=2
{'accuracy': 0.24032753326509723, 'f1': 0.17770643152899668, 'precision': 0.1498910208852314, 'recall': 0.2811188405196139, 'hamming_loss': 0.04449480918262904}
=== C=1, min_df=2
{'accuracy': 0.21699078812691913, 'f1': 0.1813461383672618, 'precision': 0.16536757277403386, 'recall': 0.3052714389458287, 'hamming_loss': 0.04717794999268899}
=== C=0.5, min_df=2
{'accuracy': 0.19242579324462641, 'f1': 0.19175120169689672, 'precision': 0.17034126406718691, 'recall': 0.3498201469386894, 'hamming_loss': 0.05030706243602866}
=== C=0.3, min_df=2
{'accuracy': 0.17154554759467758, 'f1': 0.18696797089816855, 'precision': 0.161503

In [ ]:
vec_min_df_stp = TfidfVectorizer(token_pattern=TOKEN_PATTERN, sublinear_tf=True, min_df=2, stop_words=stop_words)
A = vec_min_df_stp.fit_transform(raw_tok)
B = vec_min_df_stp.transform(test_tok)

for c in [4.6, 2.7, 2, 1, 0.5, 0.3, 0.1]:
    model = OneVsRestClassifier(LogisticRegression(solver="liblinear", C=c, class_weight="balanced", random_state=SEED)).fit(A, y_bin)
    y_pred = model.predict(B)
    print(f"=== C={c}, min_df=2, stopwords")
    print(compute_metrics(y_test, y_pred))

=== C=4.6, min_df=2, stopwords
{'accuracy': 0.2646878198567042, 'f1': 0.17877784950952472, 'precision': 0.15879035328705426, 'recall': 0.2527125729002054, 'hamming_loss': 0.0420456207047814}
=== C=2.7, min_df=2, stopwords
{'accuracy': 0.2538382804503582, 'f1': 0.17923019189904701, 'precision': 0.15537320529765464, 'recall': 0.26710480865869407, 'hamming_loss': 0.043193449334698056}
=== C=2, min_df=2, stopwords
{'accuracy': 0.24790174002047083, 'f1': 0.18206250340236746, 'precision': 0.1539302099082635, 'recall': 0.28309808955503174, 'hamming_loss': 0.04404883754934932}
=== C=1, min_df=2, stopwords
{'accuracy': 0.22231320368474924, 'f1': 0.1818423806112907, 'precision': 0.15010927078081612, 'recall': 0.31017663751148844, 'hamming_loss': 0.046812399473607255}
=== C=0.5, min_df=2, stopwords
{'accuracy': 0.20061412487205732, 'f1': 0.1801992906385998, 'precision': 0.161570216718997, 'recall': 0.3300205884867914, 'hamming_loss': 0.049371253107179415}


In [55]:
ngram_vec = TfidfVectorizer(token_pattern=TOKEN_PATTERN, ngram_range=(1, 2), sublinear_tf=True, stop_words=stop_words)
A_ngram = ngram_vec.fit_transform(raw_tok)
B_ngram = ngram_vec.transform(test_tok)

for c in [6, 4.6, 2.7, 2, 1, 0.5, 0.3, 0.1]:
    ngrams = OneVsRestClassifier(LogisticRegression(solver="liblinear", C=c, class_weight="balanced", random_state=SEED)).fit(A_ngram, y_bin)
    y_pred = ngrams.predict(B_ngram)
    print(f"=== C = {c}, ngrams (1, 2) stopwords")
    print(compute_metrics(y_test, y_pred))
    print(f"Rasgos: {A_ngram.shape[1]} n-gramas frente a {A.shape[1]} con palabras")

=== C = 6, ngrams (1, 2) stopwords
{'accuracy': 0.3344933469805527, 'f1': 0.16401552409279327, 'precision': 0.1960441394309589, 'recall': 0.19856356672169176, 'hamming_loss': 0.035999415119169466}
Rasgos: 1703642 n-gramas frente a 52005 con palabras
=== C = 4.6, ngrams (1, 2) stopwords
{'accuracy': 0.32896622313203683, 'f1': 0.16585529648130667, 'precision': 0.19281001100200362, 'recall': 0.20516435456320556, 'hamming_loss': 0.036496563825120634}
Rasgos: 1703642 n-gramas frente a 52005 con palabras
=== C = 2.7, ngrams (1, 2) stopwords
{'accuracy': 0.3127942681678608, 'f1': 0.16590531042033627, 'precision': 0.18647922593532987, 'recall': 0.215182895176352, 'hamming_loss': 0.03797338792221085}
Rasgos: 1703642 n-gramas frente a 52005 con palabras
=== C = 2, ngrams (1, 2) stopwords
{'accuracy': 0.3029682702149437, 'f1': 0.17262156947892573, 'precision': 0.189666794027203, 'recall': 0.23146750069276645, 'hamming_loss': 0.038784910074572305}
Rasgos: 1703642 n-gramas frente a 52005 con palabr

In [56]:
ngram_vec = TfidfVectorizer(token_pattern=TOKEN_PATTERN, ngram_range=(1, 2), sublinear_tf=True)
A_ngram = ngram_vec.fit_transform(raw_tok)
B_ngram = ngram_vec.transform(test_tok)

for c in [6, 4.6, 2.7, 2, 1, 0.5, 0.3, 0.1]:
    ngrams = OneVsRestClassifier(LogisticRegression(solver="liblinear", C=c, class_weight="balanced", random_state=SEED)).fit(A_ngram, y_bin)
    y_pred = ngrams.predict(B_ngram)
    print(f"=== C = {c}, ngrams (1, 2)")
    print(compute_metrics(y_test, y_pred))
    print(f"Rasgos: {A_ngram.shape[1]} n-gramas frente a {A.shape[1]} con palabras")

=== C = 6, ngrams (1, 2)
{'accuracy': 0.3465711361310133, 'f1': 0.16337766790819197, 'precision': 0.21208656308571042, 'recall': 0.18444202684043765, 'hamming_loss': 0.03486620851001609}
Rasgos: 1324815 n-gramas frente a 52005 con palabras
=== C = 4.6, ngrams (1, 2)
{'accuracy': 0.34022517911975436, 'f1': 0.16711202817570683, 'precision': 0.21110293460706822, 'recall': 0.19293072144245343, 'hamming_loss': 0.03536335721596725}
Rasgos: 1324815 n-gramas frente a 52005 con palabras
=== C = 2.7, ngrams (1, 2)
{'accuracy': 0.32384851586489255, 'f1': 0.16545616343906344, 'precision': 0.2039280883274768, 'recall': 0.19856435763966657, 'hamming_loss': 0.03687673636496564}
Rasgos: 1324815 n-gramas frente a 52005 con palabras
=== C = 2, ngrams (1, 2)
{'accuracy': 0.31238485158648926, 'f1': 0.16917762172264933, 'precision': 0.20306868667751468, 'recall': 0.2096036370810205, 'hamming_loss': 0.03784178973534143}
Rasgos: 1324815 n-gramas frente a 52005 con palabras
=== C = 1, ngrams (1, 2)
{'accuracy

In [57]:
ngram_vec = TfidfVectorizer(token_pattern=TOKEN_PATTERN, ngram_range=(1, 2), sublinear_tf=True, min_df=2)
A_ngram = ngram_vec.fit_transform(raw_tok)
B_ngram = ngram_vec.transform(test_tok)

for c in [6, 4.6, 2.7, 2, 1, 0.5, 0.3, 0.1]:
    ngrams = OneVsRestClassifier(LogisticRegression(solver="liblinear", C=c, class_weight="balanced", random_state=SEED)).fit(A_ngram, y_bin)
    y_pred = ngrams.predict(B_ngram)
    print(f"=== C = {c}, ngrams (1, 2), min_df=2")
    print(compute_metrics(y_test, y_pred))
    print(f"Rasgos: {A_ngram.shape[1]} n-gramas frente a {A.shape[1]} con palabras")

=== C = 6, ngrams (1, 2), min_df=2
{'accuracy': 0.3428863868986694, 'f1': 0.16147172963543419, 'precision': 0.2078285705160748, 'recall': 0.18302652453521048, 'hamming_loss': 0.03520251498757128}
Rasgos: 631277 n-gramas frente a 52005 con palabras
=== C = 4.6, ngrams (1, 2), min_df=2
{'accuracy': 0.33469805527123847, 'f1': 0.16504388776515358, 'precision': 0.20888929282569005, 'recall': 0.19040058110544228, 'hamming_loss': 0.035794706828483694}
Rasgos: 631277 n-gramas frente a 52005 con palabras
=== C = 2.7, ngrams (1, 2), min_df=2
{'accuracy': 0.3195496417604913, 'f1': 0.17031342007870104, 'precision': 0.20137867356050818, 'recall': 0.20658403850654822, 'hamming_loss': 0.037103377686796314}
Rasgos: 631277 n-gramas frente a 52005 con palabras
=== C = 2, ngrams (1, 2), min_df=2
{'accuracy': 0.3082906857727738, 'f1': 0.17179567228623055, 'precision': 0.19524656785230238, 'recall': 0.2176284318062322, 'hamming_loss': 0.03815616318175172}
Rasgos: 631277 n-gramas frente a 52005 con palabras

In [58]:
ngram_vec = TfidfVectorizer(token_pattern=TOKEN_PATTERN, ngram_range=(1, 2), sublinear_tf=True, min_df=2, stop_words=stop_words)
A_ngram = ngram_vec.fit_transform(raw_tok)
B_ngram = ngram_vec.transform(test_tok)

for c in [6, 4.6, 2.7, 2, 1, 0.5, 0.3, 0.1]:
    ngrams = OneVsRestClassifier(LogisticRegression(solver="liblinear", C=c, class_weight="balanced", random_state=SEED)).fit(A_ngram, y_bin)
    y_pred = ngrams.predict(B_ngram)
    print(f"=== C = {c}, ngrams (1, 2), min_df=2, stopwords")
    print(compute_metrics(y_test, y_pred))
    print(f"Rasgos: {A_ngram.shape[1]} n-gramas frente a {A.shape[1]} con palabras")

=== C = 6, ngrams (1, 2), min_df=2, stopwords
{'accuracy': 0.32814738996929377, 'f1': 0.17589205252846699, 'precision': 0.20111696909855437, 'recall': 0.21263194629148657, 'hamming_loss': 0.036204123409855245}
Rasgos: 752836 n-gramas frente a 52005 con palabras
=== C = 4.6, ngrams (1, 2), min_df=2, stopwords
{'accuracy': 0.3218014329580348, 'f1': 0.17576316107464457, 'precision': 0.19912058816069628, 'recall': 0.21726137578790428, 'hamming_loss': 0.036869425354584005}
Rasgos: 752836 n-gramas frente a 52005 con palabras
=== C = 2.7, ngrams (1, 2), min_df=2, stopwords
{'accuracy': 0.30337768679631527, 'f1': 0.17998366106472816, 'precision': 0.1978493200863936, 'recall': 0.2340634776477987, 'hamming_loss': 0.03841935955549057}
Rasgos: 752836 n-gramas frente a 52005 con palabras
=== C = 2, ngrams (1, 2), min_df=2, stopwords
{'accuracy': 0.29293756397134085, 'f1': 0.18429079954564087, 'precision': 0.19866207852198006, 'recall': 0.249556965520847, 'hamming_loss': 0.039450212019301066}
Rasgos

In [76]:
def load_corpus(filename, is_test=False, delimiter=','):
    texts, terms = [], []
    with open(filename, mode='r', encoding='utf-8') as file:
        reader = csv.reader(file, delimiter=delimiter if not is_test else ';')
        header = next(reader)
        for row in reader:
            if not row or len(row) < 3:
                continue
            title, summary = row[0], row[1]
            keywords = row[2] if (is_test and len(row) > 2) else ""
            
            # Concatenamos título, resumen y keywords si existen
            text_combined = f"{title} . {summary} . {keywords}".strip()
            texts.append(text_combined)
            
            if not is_test:
                terms.append(ast.literal_eval(row[2]))
            else:
                if len(row) > 3 and row[3].startswith('['):
                    terms.append(ast.literal_eval(row[3]))
                else:
                    terms.append([])
    return texts, terms

def compute_metrics(labels, preds):
    precision, recall, f1, _ = precision_recall_fscore_support(
        labels, preds, average="macro", zero_division=0
    )
    acc = accuracy_score(labels, preds)
    hl = hamming_loss(labels, preds)
    return {
        "accuracy": acc,
        "f1": f1,
        "precision": precision,
        "recall": recall,
        "hamming_loss": hl,
    }

In [ ]:
# Carga de datos
train_texts_raw, train_terms = load_corpus(TRAIN_FILE, is_test=False)
mlb = MultiLabelBinarizer()
y_all = mlb.fit_transform(train_terms)

X_train_raw, X_dev_raw, y_train, y_dev = train_test_split(
    train_texts_raw, y_all, test_size=DEV_PART, random_state=SEED, shuffle=True
)

# ==========================================
# 2. MODELO CLÁSICO: TF-IDF + LINEARSVC
# ==========================================

vec = TfidfVectorizer(ngram_range=(1, 2), max_features=40000, sublinear_tf=True, stop_words='english')
X_train_vec = vec.fit_transform(X_train_raw)
X_dev_vec = vec.transform(X_dev_raw)

svm_model = OneVsRestClassifier(
    LinearSVC(C=0.5, class_weight="balanced", random_state=SEED, max_iter=5000),
    n_jobs=-1
)
svm_model.fit(X_train_vec, y_train)
preds_dev_svm = svm_model.predict(X_dev_vec)

print("=== Resultados Modelo Clásico (LinearSVC) en Validación ===")
print(compute_metrics(y_dev, preds_dev_svm))

In [77]:
X_test, y_test_raw = load_corpus("data/etiquetas_test.csv", True, delimiter=';')
y_test = mlb.transform(y_test_raw)
X_test_vec = vec.transform(X_test)
preds_test_svm = svm_model.predict(X_test_vec)
print(compute_metrics(y_test, preds_test_svm))

{'accuracy': 0.3127942681678608, 'f1': 0.1771491046767048, 'precision': 0.22493516300571717, 'recall': 0.2052279613030714, 'hamming_loss': 0.037242286884047376}


# Transformers #

In [ ]:
from transformers import (AutoTokenizer, AutoModelForSequenceClassification,
                          Trainer, TrainingArguments, EarlyStoppingCallback, set_seed, default_data_collator)
import torch
import pandas as pd

In [ ]:
MODEL_NAME = "dccuchile/bert-base-spanish-wwm-uncased"

In [ ]:
PIPELINE_ARGS = {"num_train_epochs", "learning_rate", "per_device_train_batch_size", "per_device_eval_batch_size",
                 "warmup_steps", "weight_decay", "eval_strategy", "save_strategy", "load_best_model_at_end",
                 "metric_for_best_model", "early_stopping_patience", "max_len", "output_dir", "seed", "compute_metrics"}

def classification_pipeline_task1(trainInfo, devInfo, testInfo=None, model_name=MODEL_NAME, nlabels=4,
                                  ptype="single_label_classification", **args):
    """Ajuste fino de un modelo encoder para clasificar secuencias.

    trainInfo, devInfo, testInfo: tuplas (textos, etiquetas codificadas); devInfo y testInfo pueden ser None
    (sin desarrollo no se evalúa durante el entrenamiento ni se elige el mejor modelo: se usa el de la última epoch).
    ptype: "single_label_classification" (una clase por texto) o "multi_label_classification".
    **args: hiperparámetros opcionales (ver PIPELINE_ARGS).
    Devuelve el Trainer, con el mejor modelo ya cargado, y un diccionario con los resultados.
    """
    unknown = set(args) - PIPELINE_ARGS
    if unknown:
        raise ValueError(f"Argumentos desconocidos: {sorted(unknown)}")
    seed = args.get("seed", SEED)
    set_seed(seed)

    # Modelo y tokenizador. La cabeza de clasificación (nlabels salidas) se inicializa al azar.
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=nlabels, problem_type=ptype)

    multilabel = (ptype == "multi_label_classification")
    max_len = args.get("max_len", MAX_LEN)
    train_dataset = TweetDataset(trainInfo[0], trainInfo[1], tokenizer, max_len, multilabel)
    dev_dataset = None if devInfo is None else TweetDataset(devInfo[0], devInfo[1], tokenizer, max_len, multilabel)
    with_dev = devInfo is not None

    training_args = TrainingArguments(
        output_dir=args.get("output_dir", "./results"),
        report_to="none",                                   
        num_train_epochs=args.get("num_train_epochs", 5),
        learning_rate=args.get("learning_rate", 5e-5),
        per_device_train_batch_size=args.get("per_device_train_batch_size", 64),
        per_device_eval_batch_size=args.get("per_device_eval_batch_size", 64),
        warmup_steps=args.get("warmup_steps", 0.1),# entero: pasos; entre [0, 1): proporción del total
        weight_decay=args.get("weight_decay", 0.01),
        eval_strategy=args.get("eval_strategy", "epoch" if with_dev else "no"),
        logging_strategy="epoch",
        save_strategy=args.get("save_strategy", "best" if with_dev else "no"),
        save_total_limit=1, save_only_model=True,
        load_best_model_at_end=args.get("load_best_model_at_end", with_dev),
        metric_for_best_model=args.get("metric_for_best_model", "f1"),
        seed=seed,
    )

    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=train_dataset,
        eval_dataset=dev_dataset,
        processing_class=tokenizer,
        compute_metrics=args.get("compute_metrics", compute_metrics),
        callbacks=[EarlyStoppingCallback(early_stopping_patience=args.get("early_stopping_patience", 3))] if with_dev else [],
        data_collator=default_data_collator
    )

    time0 = time.time()
    trainer.train()
    results = trainer.evaluate() if with_dev else {}   # métricas en desarrollo del mejor modelo
    results["epoch"] = trainer.state.epoch              # en transformers 5, evaluate() ya no la devuelve
    results["train_time"] = time.time() - time0

    if testInfo is not None:
        test_dataset = TweetDataset(testInfo[0], testInfo[1], tokenizer, max_len, multilabel)
        results["test_logits"] = trainer.predict(test_dataset).predictions   # una fila de logits por texto
    return trainer, results

In [ ]:
params = {
    "num_train_epochs": 5,
    "learning_rate": 5e-5,
    "per_device_train_batch_size": 64,
    "metric_for_best_model": "f1",
}
trainer, res = classification_pipeline_task1((train_tok, y_train), (dev_tok, y_dev), (test_tok, y_test),
                                             MODEL_NAME, len(mlb.classes_), "single_label_classification", **params)

log = pd.DataFrame(trainer.state.log_history)
dev_log = log.dropna(subset=["eval_loss"]).drop_duplicates("epoch")[["epoch", "eval_loss", "eval_f1"]]
best_epoch = int(dev_log.loc[dev_log.eval_f1.idxmax(), "epoch"])

print(f"Entrenamiento: {res['train_time'] / 60:.1f} min ({res['train_time'] / res['epoch']:.0f} s por epoch)")
print(f"Mejor epoch (mayor F1 macro en validación): {best_epoch}   F1 macro {res['eval_f1']:.4f}   pérdida {res['eval_loss']:.4f}")

Evaluación en validación

In [ ]:
pred_dev = mlb.inverse_transform(trainer.predict(trainer.eval_dataset).predictions.argmax(-1))
res_dev = compute_metrics(y_dev, pred_dev, name=f"BETO, epoch {best_epoch} (validación)")

Evaluación en test

In [ ]:
pred_test = mlb.inverse_transform(res["test_logits"].argmax(-1))
res_test = compute_metrics(y_test, pred_test, name=f"BETO, epoch {best_epoch} (test)")

¿Cómo contar cuántas muestras se quedan sin ninguna etiqueta?

### Prueba con transformer scibert ###

In [85]:
X_test, y_test = load_corpus("data/etiquetas_test.csv", delimiter=';')
X_pred, y_pred = load_corpus("dataset_test_preds.csv")

y_test_bin = mlb.transform(y_test)

y_pred_bin = mlb.transform(y_pred)

print(compute_metrics(y_test_bin, y_pred_bin))

{'accuracy': 0.3265097236438076, 'f1': 0.18844878380757915, 'precision': 0.1916594751684402, 'recall': 0.23124135260855766, 'hamming_loss': 0.036079836233367454}
